# 01 · Multimodal Data Processing of Autonomous Driving Images

> **阿里云 PAI DSW · nuScenes CAM_FRONT · DLF 2.0 / PyPaimon · GPT Caption/Metadata · 多模态 Embedding · 图文检索**

这是**可逐 Cell 执行**的实验教程。由阿里云多模态数据处理教程的 Notebook 组织形式改写，底层使用我们已实践的 **DSW + PyPaimon REST** 路线，不使用原教程的 EMR Spark。复杂实现封装在旁边的 Python 模块，Notebook 负责讲解、操作和显示中间结果。

**建议顺序**：Step 0 环境准备 → Step 1–3 初始化与数据 → Step 4–7 Paimon/Caption → Step 8–10 Embedding 与检索。

**费用与安全**：下载约 GB 级数据、运行视觉 API / Embedding、Paimon 写入及 GPU 都可能产生费用。涉及网络调用或写入的 Cell 都有显式开关。不要上传 `.env`、训练样本、密钥或 API 响应中可能含有的认证头。

## Step 0 · 环境准备与 Kernel 重启

首次运行请在 DSW **Terminal** 中执行（已有仓库则跳过 `git clone`）：

```bash
git clone https://github.com/sxxa-qlp/aliyun-autonomous-driving-multimodal-pipeline.git
cd aliyun-autonomous-driving-multimodal-pipeline
cp -n .env.example .env
# 编辑 .env：填入阿里云 RAM AK/SK、视觉模型和百炼 Embedding 的 API 地址/Key
# 先根据 README 2.5 核查 CUDA 12.6 / torch==2.9.1+cu126 与 V100 是否兼容
python -m pip install -r requirements-data.txt -c constraints-v100.txt
python -m pip install -r requirements-train.txt -c constraints-v100.txt
python -m pip check
```

**安装完成后：在 Notebook 菜单点击 Kernel → Restart Kernel**，随后从 Step 1 顺序运行。`config.py` 会自动加载项目根目录的 `.env`。千万不要直接复制阿里云官方案例中的 `--force-reinstall pypaimon`，它曾导致 V100 上的 CUDA 环境失效。

**重启只清空内存变量**，不会删除 `/mnt/systemDisk` 上的文件和已经保存的 Paimon 表。

## Step 1 · 进入项目目录 / 初始化 Python

In [ ]:
# Step 1：定位仓库根目录；无论 Notebook 从根目录还是 notebooks/ 打开都适用
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / 'aliyun-autonomous-driving-multimodal-pipeline',
              Path.home() / 'workspace' / 'aliyun-autonomous-driving-multimodal-pipeline']
ROOT = next((p for p in candidates if (p / 'config.py').is_file()
             and (p / 'multimodal_data_processing').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('未找到项目根目录。请先 git clone，并在仓库的 notebooks/ 目录打开本教程。')
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
print('Project root:', ROOT)

## Step 2 · 检查 Python、GPU 与 `.env` 参数

`DLF credentials: configured` 仅表示两个环境变量非空；实际读写权限要在 Step 4 验证。

In [ ]:
# Step 2：读取 .env → Python 配置；不输出任何密钥原文
from config import settings, check_environment

check_environment()
print('DLF Warehouse:', settings.dlf_warehouse)
print('Raw / Enriched:', settings.raw_table, settings.enriched_table)
print('Work dir:', settings.work_dir)
print('DSW 配置检查:', {
    'DLF AK/SK': bool(settings.access_key_id and settings.access_key_secret),
    'Vision': bool(settings.vision_base_url and settings.vision_key and settings.vision_model != 'your-vision-model'),
    'Embedding': bool(settings.embedding_url and settings.embedding_key),
})

## Step 3 · 下载 nuScenes mini / 选择 CAM_FRONT 连续帧

原始数据来自 nuScenes。需要先遵守 [nuScenes 官方数据使用条款](https://www.nuscenes.org/)。本实验希望挑选同一段驾驶日志的 **20 张连续 CAM_FRONT 图像**。

下载和解压耗时、占用云盘。已经存在 `settings.nu_scenes_dir` 或 Paimon 原始表时，可以直接跳到下一节。

In [ ]:
# 首次下载前先确认数据集许可；默认不发起网络请求
RUN_DOWNLOAD = False  # 阅读 nuScenes 许可后改 True，仅运行此 Cell 一次

if RUN_DOWNLOAD:
    from multimodal_data_processing.download_nuscenes import download_archive, extract_safe, URL
    archive = download_archive(URL, settings.work_dir / 'v1.0-mini.tgz')
    extract_safe(archive, settings.nu_scenes_dir)
    print('数据解压完成:', settings.nu_scenes_dir)
else:
    print('跳过下载；已有数据请继续下一步。')

In [ ]:
# 扫描本地 CAM_FRONT；没有本地文件也可以继续使用已存在的 Paimon 表
from multimodal_data_processing.prepare_images import list_camfront_images, select_adjacent, read_images
from PIL import Image
from IPython.display import display

selected = []
for source in ('sweeps', 'samples'):
    paths = list_camfront_images(settings.nu_scenes_dir, source=source)
    try:
        selected = select_adjacent(paths, limit=20)
        print('数据源:', source, '候选数:', len(paths), '选择:', len(selected))
        break
    except ValueError as err:
        print(source, ':', str(err)[:150])

if selected:
    print('第一张:', selected[0].name)
    with Image.open(selected[0]) as im:
        display(im.convert('RGB').resize((640, 360)))
else:
    print('未选到足够的连续 CAM_FRONT：如果之前已写入 Paimon，直接从 Step 4 读取。')

## Step 4 · 连接 DLF 2.0 / 查询 Paimon 表

关键数据流：**DSW → DLF 2.0 REST Catalog → Paimon**。这里是真实的权限和网络连通性检查，至少需要 `.env` 配置 AK/SK、Warehouse 与 Endpoint。

In [ ]:
from multimodal_data_processing.paimon_io import (
    connect_catalog, read_table, ensure_table, RAW_SCHEMA, ENRICHED_SCHEMA,
    upload_raw_images,
)

catalog = connect_catalog()
print('Databases:', catalog.list_databases())
print('Default tables:', catalog.list_tables('default'))

## Step 5 · 原始图像写入 Paimon，并重新读取确认

目标表：`default.camfront_raw_images`，字段 `fileName` 和 `picture`。为防止重复写入，模块在表非空时会跳过 Append。**已有 20 张图像的用户不用再次写入。**

In [ ]:
# 跨网络写入需明确授权；已存在表不必再次上传
RUN_RAW_WRITE = False

if RUN_RAW_WRITE:
    assert selected, '先完成 Step 3 的本地图像选择'
    written = upload_raw_images(read_images(selected))
    print('本次新增写入:', written)
else:
    print('跳过写入；下面将读取已有表。')

raw_table = catalog.get_table(settings.raw_table)
raw_df = read_table(raw_table)
print('Paimon 原始图像行数:', len(raw_df))
if not raw_df.empty:
    display(raw_df[['fileName']].head())
    from io import BytesIO
    with Image.open(BytesIO(bytes(raw_df.iloc[0]['picture']))) as im:
        display(im.convert('RGB').resize((640, 360)))

## Step 6 · 视觉大模型生成 Caption 与结构化 Metadata

采用 **OpenAI Responses 兼容接口**：一次调用生成 1–2 句英文 Caption，以及 weather / lighting / road_type / scene_type / objects / risks 等字段。运行前确保 `.env` 配置 `VISION_API_BASE_URL`、`VISION_API_KEY`、`VISION_MODEL`。

标签结果保存到 `settings.caption_csv`，支持按 `fileName` 断点续跑；**API 请求会计费**。

In [ ]:
from multimodal_data_processing.caption_metadata import annotate_to_csv
import pandas as pd

RUN_CAPTION_LABELING = False  # 首次标注时改 True；已有 CSV 时保持 False
if RUN_CAPTION_LABELING:
    assert not raw_df.empty, 'Paimon 原始表为空'
    annotated_count = annotate_to_csv(raw_df.head(20).to_dict('records'), settings.caption_csv)
    print('本次新增标注:', annotated_count)
else:
    print('跳过视觉 API 请求（已有结果会从本地 CSV 读取）。')

if settings.caption_csv.is_file():
    labels_df = pd.read_csv(settings.caption_csv)
    display(labels_df[['fileName','caption','weather','lighting','road_type']].head())
else:
    labels_df = None
    print('未找到 Caption CSV。需要标注时把 RUN_CAPTION_LABELING 改 True。')

## Step 7 · 创建增强 Paimon 表

目标表 `default.camfront_ai_enriched` 增加 `caption`、`metadata`、`vector`、`dataset_tag=1` 等字段。这里采用**无主键 Parquet Append 表**，不是原教程的 Lance 表。创建与首批写入是幂等设计，已有记录时跳过。

In [ ]:
from multimodal_data_processing.enrichment import create_enriched_table

RUN_CREATE_ENRICHED = False
if RUN_CREATE_ENRICHED:
    assert labels_df is not None, '先完成 Caption/Metadata 标注'
    create_enriched_table()
else:
    print('跳过增强表创建/写入')

if settings.enriched_table.split('.', 1)[1] in catalog.list_tables('default'):
    enriched_table = catalog.get_table(settings.enriched_table)
    enriched_df = read_table(enriched_table)
    print('增强表行数:', len(enriched_df))
    if not enriched_df.empty:
        display(enriched_df[['fileName','caption','weather','lighting','road_type']].head())
else:
    enriched_df = pd.DataFrame()
    print('增强表尚不存在；执行上方创建 Cell（RUN_CREATE_ENRICHED=True）。')

## Step 8 · 生成 1152 维多模态 Embedding

使用百炼 **DashScope 原生多模态 Embedding API** 的 `tongyi-embedding-vision-plus`。图像向量和文本向量分别生成，但位于兼容的表示空间。向量缓存按内容哈希存储在 DSW 云盘；**首次调用 API 可能计费**。

In [ ]:
from multimodal_data_processing.embedding import image_embedding, text_embedding

RUN_SAMPLE_EMBEDDING = False
if RUN_SAMPLE_EMBEDDING:
    assert not enriched_df.empty, '增强表不存在'
    vec = image_embedding(bytes(enriched_df.iloc[0]['picture']))
    print('第一张图片的向量形状:', vec.shape)
    print('向量前 8 维:', vec[:8])
else:
    print('跳过单张图像 Embedding API 测试')

## Step 9 · 向量回填 Paimon（谨慎操作）

当前增强表是**无主键 Append 表**，不能通过再次 Append 相同 ID 的记录实现字段更新。实际实现先备份整表，再生成向量，最后明确执行 `overwrite()`。**不要在其他写入任务并发运行时操作。**

如果表已有 1152 维向量，函数会自动跳过；如果只回填了一部分，会报错提醒手工检查。

In [ ]:
from multimodal_data_processing.enrichment import backfill_vectors

RUN_VECTOR_BACKFILL = False  # 修改为 True 前确认没有并发写入且已备份方案
if RUN_VECTOR_BACKFILL:
    backup_path = backfill_vectors(confirm_overwrite=True)
    print('备份文件:', backup_path)
else:
    print('未执行整表回填（默认安全跳过）')

if settings.enriched_table.split('.', 1)[1] in catalog.list_tables('default'):
    enriched_df = read_table(catalog.get_table(settings.enriched_table))
    if not enriched_df.empty:
        print('当前记录数:', len(enriched_df))
        print('首条 vector 是否非空:', enriched_df.iloc[0]['vector'] is not None)

## Step 10 · 以图搜图 / 以文搜图：Cosine Top-5

这里使用 NumPy 精确余弦相似度检索 20 张图像，**不依赖 Milvus / ANN 索引**。以图搜图可直接用表中已保存的向量；以文搜图则额外调用一次文本 Embedding API。

In [ ]:
import numpy as np
from multimodal_data_processing.retrieval import cosine_topk

if enriched_df.empty or any(v is None for v in enriched_df.get('vector', [])):
    print('请先完成 Step 9：当前没有完整的图像向量。')
else:
    matrix = np.stack([np.asarray(v, dtype=np.float32) for v in enriched_df['vector']])
    query_idx = 0
    hits = cosine_topk(matrix, matrix[query_idx], k=5, exclude_index=query_idx)
    search_rows = [{'rank': rank, 'similarity': round(score, 4),
                    'fileName': enriched_df.iloc[idx]['fileName'],
                    'caption': enriched_df.iloc[idx]['caption']}
                   for rank, (idx, score) in enumerate(hits, 1)]
    print('以图搜图 Top-5：')
    display(pd.DataFrame(search_rows))
    with Image.open(BytesIO(bytes(enriched_df.iloc[hits[0][0]]['picture']))) as im:
        display(im.convert('RGB').resize((640, 360))) if hits else None

In [ ]:
# 以文搜图：需向百炼发送文本 Embedding API 请求
RUN_TEXT_RETRIEVAL = False
query = 'A cloudy daytime urban intersection with pedestrians, cars and traffic lights.'
if RUN_TEXT_RETRIEVAL:
    assert not enriched_df.empty and all(v is not None for v in enriched_df['vector'])
    text_vector = text_embedding(query)
    matrix = np.stack([np.asarray(v, dtype=np.float32) for v in enriched_df['vector']])
    hits = cosine_topk(matrix, text_vector, k=5)
    display(pd.DataFrame([
        {'rank': rank, 'similarity': round(score, 4),
         'fileName': enriched_df.iloc[idx]['fileName'],
         'caption': enriched_df.iloc[idx]['caption']}
        for rank, (idx, score) in enumerate(hits, 1)
    ]))
else:
    print('默认跳过文本 Embedding 请求')

## ✅ Part 1 完成检查

完成后，`default.camfront_ai_enriched` 应包含非空 Caption、结构化 Metadata、1152 维图像向量以及 `dataset_tag=1`。**下一篇 Notebook 02** 将读取该表（也可直接复用已导出的 `train.jsonl` / `validation.jsonl`），完成 Qwen2-VL NF4 量化、QLoRA 微调与同图对比。

[打开 Notebook 02](02_multimodal_model_training_inference.ipynb) · [查看 Part 1 模块源码](../multimodal_data_processing/) · [返回项目 README](../README.md)

> 不要把此次 20 张相似连续帧的检索/训练结果解释成大规模数据集的性能验证。